### Установка Unsloth и TRL (для запуска в Colab / на GPU)

In [ ]:
# !pip install unsloth vllm
# !pip install --no-deps trl peft accelerate bitsandbytes

### : Инициализация модели Qwen2.5 + QLoRA (4-bit)

In [ ]:
import re
import json
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer

MAX_SEQ_LENGTH = 1024
LORA_RANK = 16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-1.5B-Instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,         # QLoRA 4-bit для экономии видеопамяти
    fast_inference=True,       # Поднимает vLLM (PagedAttention) для генерации групп в GRPO
    max_lora_rank=LORA_RANK,
    gpu_memory_utilization=0.6,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_RANK,
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

### Кастомные детерминированные Reward-функции (Rule-Based Verifiers)

In [ ]:
def format_reward_func(completions, **kwargs) -> list[float]:
    """Reward 1: Награда за строгое следование формату рассуждений <think>...</think><answer>...</answer>."""
    pattern = r"^<think>.*?</think>\s*<answer>.*?</answer>$"
    return [
        0.5 if re.match(pattern, c[0]["content"].strip(), re.DOTALL) else 0.0
        for c in completions
    ]


def json_schema_reward_func(completions, **kwargs) -> list[float]:
    """Reward 2: Награда за валидный JSON с ключами оценки риска внутри <answer>."""
    rewards = []
    for c in completions:
        match = re.search(r"<answer>(.*?)</answer>", c[0]["content"], re.DOTALL)
        if not match:
            rewards.append(0.0)
            continue
        try:
            data = json.loads(match.group(1).strip())
            rewards.append(0.5 if ("is_compliant" in data and "severity" in data) else 0.1)
        except Exception:
            rewards.append(0.0)
    return rewards


def temporal_accuracy_reward_func(completions, expected_compliant, expected_severity, **kwargs) -> list[float]:
    """Reward 3: Награда за правильный вердикт по операционному риску с учетом версии регламента."""
    rewards = []
    for c, exp_comp, exp_sev in zip(completions, expected_compliant, expected_severity):
        match = re.search(r"<answer>(.*?)</answer>", c[0]["content"], re.DOTALL)
        if not match:
            rewards.append(0.0)
            continue
        try:
            data = json.loads(match.group(1).strip())
            score = 0.0
            if data.get("is_compliant") == exp_comp:
                score += 1.0
            if data.get("severity") == exp_sev:
                score += 1.0
            rewards.append(score)
        except Exception:
            rewards.append(0.0)
    return rewards

In [ ]:
SYSTEM_PROMPT = (
    "Ты — AI-аудитор операционных рисков банка. "
    "Сначала проведи сверку дат и лимитов внутри тегов <think>...</think>, "
    "а затем выдай финальный вердикт внутри тегов <answer>...</answer> в формате JSON: "
    '{"is_compliant": bool, "severity": "LOW" | "HIGH" | "CRITICAL"}.'
)

train_data = Dataset.from_list([
    {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Дата процесса: 2026-03-15. Регламент v2.0 (с 2025-01-01): Лимит = 100 000 руб. Процесс: автосписание до 350 000 руб."}
        ],
        "expected_compliant": False,
        "expected_severity": "CRITICAL",
    },
    {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Дата процесса: 2024-06-10. Регламент v1.2 (2023..2024): Лимит = 500 000 руб. Регламент v2.0 (с 2025): Лимит = 100 000 руб. Процесс: автосписание до 350 000 руб."}
        ],
        "expected_compliant": True,
        "expected_severity": "LOW",
    },
])

training_args = GRPOConfig(
    output_dir="outputs/qwen-risk-grpo",
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,          # G=4 сэмпла на промпт для оценки Advantage без Value-модели
    max_prompt_length=256,
    max_completion_length=512,
    num_train_epochs=1,
    logging_steps=1,
    bf16=True,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        format_reward_func,
        json_schema_reward_func,
        temporal_accuracy_reward_func,
    ],
    args=training_args,
    train_dataset=train_data,
)

# trainer.train()